# 04 · Stage 3 — TreeSHAP explainability
Exact TreeSHAP for the selected model, additivity check, national five-domain attribution and a local district explanation (blueprint §3.3).

In [1]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.configuration import load_config, configured_path, feature_columns, zones
config = load_config()
P = lambda name: configured_path(config, name)
pd.set_option("display.max_columns", 40)

In [2]:
import joblib
from src.explain.shap_profiler import explain, domain_contributions, national_domain_shares
model = joblib.load(P("model"))
features = list(model.named_steps["imputer"].feature_columns)
master = pd.read_csv(P("master_dataset"))
modelling = master.loc[master["priority_class"].notna()].reset_index(drop=True)
modelling["priority_class"] = modelling["priority_class"].astype(int)
result = explain(model, modelling[["state", *features]])
print("Output space:", result["output_space"], "| base value:", round(result["base_value"], 4),
      "| max additivity error:", f"{result['max_additivity_error']:.2e}")

Output space: log_odds | base value: -0.2154 | max additivity error: 5.33e-15


In [3]:
import shap
shap.summary_plot(result["values"], pd.DataFrame(result["transformed"], columns=features), max_display=15, show=False)
plt.tight_layout(); plt.show()

In [4]:
domains = domain_contributions(result["values"], features)
national_domain_shares(domains, (modelling["priority_class"] == 1).to_numpy())

,domain,share_pct,mean_signed_contribution,scheme
0,maternal_health_nutrition,44.24,1.009237,POSHAN Abhiyaan & Anaemia Mukt Bharat
1,healthcare_access_delivery,21.41,0.452923,PMSMA (9th of every month high-risk ANC) & Jan...
2,emergency_facility_quality,18.58,0.363366,LaQshya & NHM FRU/Blood Bank Upgradation
3,infrastructure_wash,8.78,0.148472,Jal Jeevan Mission & Swachh Bharat / IPHS PHC ...
4,socioeconomic_education,6.98,0.134567,Beti Bachao Beti Padhao & PM-JAY Ayushman Bharat


In [5]:
actions = pd.read_csv(P("priority_table"))
actions.loc[actions["priority_class"] == 1, ["rank", "district", "state", "priority_risk_score", "primary_domain_bottleneck", "top_1_driver", "top_2_driver"]].head(15)

,rank,district,state,priority_risk_score,primary_domain_bottleneck,top_1_driver,top_2_driver
0,1,Tiruvannamalai,Tamil Nadu,100.0,maternal_health_nutrition,hmis_severe_anaemia_pct: 6.75 (state median 6....,married_before_18_pct: 11.60 (state median 12....
1,2,Viluppuram,Tamil Nadu,100.0,maternal_health_nutrition,hmis_severe_anaemia_pct: 7.32 (state median 6....,hmis_ifa_180_pct: 77.83 (state median 93.91; S...
2,3,Mahendragarh,Haryana,96.4,maternal_health_nutrition,hmis_severe_anaemia_pct: 8.17 (state median 5....,married_before_18_pct: 13.40 (state median 11....
3,4,Dindigul,Tamil Nadu,91.1,maternal_health_nutrition,hmis_severe_anaemia_pct: 6.22 (state median 6....,hmis_early_discharge_pct: 0.01 (state median 0...
4,5,Khargone (West Nimar),Madhya Pradesh,88.2,maternal_health_nutrition,oop_expenditure_log: 3.22 (state median 3.19; ...,hmis_severe_anaemia_pct: 4.68 (state median 4....
5,6,Krishnagiri,Tamil Nadu,87.7,maternal_health_nutrition,hmis_severe_anaemia_pct: 7.74 (state median 6....,hmis_ifa_180_pct: 88.60 (state median 93.91; S...
6,7,Tiruchirappalli,Tamil Nadu,85.5,maternal_health_nutrition,hmis_severe_anaemia_pct: 5.67 (state median 6....,married_before_18_pct: 13.50 (state median 12....
7,8,Pudukkottai,Tamil Nadu,84.2,maternal_health_nutrition,hmis_severe_anaemia_pct: 7.37 (state median 6....,married_before_18_pct: 11.60 (state median 12....
8,9,Panipat,Haryana,77.4,maternal_health_nutrition,hmis_severe_anaemia_pct: 5.72 (state median 5....,hmis_ifa_180_pct: 47.15 (state median 81.67; S...
9,10,Bhind,Madhya Pradesh,74.2,maternal_health_nutrition,hmis_severe_anaemia_pct: 12.62 (state median 4...,oop_expenditure_log: 3.10 (state median 3.19; ...


In [6]:
top = actions.loc[actions["priority_class"] == 1].iloc[0]
i = modelling.index[(modelling["district"] == top["district"]) & (modelling["state"] == top["state"])][0]
explanation = shap.Explanation(values=result["values"][i], base_values=result["base_value"],
                               data=result["transformed"][i], feature_names=features)
shap.plots.waterfall(explanation, max_display=12, show=False)
plt.title(f"{top['district']}, {top['state']}"); plt.tight_layout(); plt.show()